# Exploring DuckDB's R API

## Technical requirements 

# Instructions for setting up your environment

To use DuckDB with R, you'll first need to have R installed on your system. If you don't have R installed yet, you can download it from the Comprehensive R Archive Network (CRAN) at https://cran.r-project.org/.
If you're using a Debian-based Linux distribution, you can install R using the following commands:
```bash
    sudo apt update && sudo apt install r-base
```

If you're using macOS, you can install R using Homebrew with the following command:
```bash
    brew install --cask r
```

If you're using Windows and have winget installed, you can install R by running the following command in an elevated command prompt:
```ps    
    winget install R.Project.R
```
 


## Installing R dependencies

In order to run the examples in this notebook, you'll need to install the R
dependencies for this project. You can do this by running the following command
in your R session.

To activate your R environment and run:
```bash
    R
```

In [ ]:
install.packages(c("duckdb", "tidyverse", "arrow"))

For complete instructions on how to set up your environment for working through
the examples, please consult the *Technical requirements* section of this
chapter in the book.

## Working with DuckDB using R’s DBI

In [1]:
library(duckdb)

Loading required package: DBI



The DuckDB R client provides excellent support for the R DBI (https://dbi.r-dbi.org), a standardized API for communicating with DBMS. It's the standard way of connecting to and querying from databases in R. It is similar in design to the Python DB-API, with the DBI package aiming to provide a single standardized database interface across the R ecosystem. This enables both R users and R package developers to work with and build against a single interface, rather than having to target a separate API for each target database.

The DBI package contains an extensive range of functions; you can find the complete reference at https://dbi.r-dbi.org/reference. The core functions that you will want to be comfortable with are as follows:

- **DBI::dbconnect():** Create a new connection to a target database
- **DBI::dbGetQuery():** Run a SQL query that retrieves a result set and returns those results
- **DBI::dbExecute():** Execute a SQL command that does not retrieve results against a database connection
- **DBI::dbDisconnect():** Disconnect an existing connection object to a database

### Connecting to DuckDB

To interact with DuckDB in R, we need to create a connection to a DuckDB database using the **DBI::dbConnect()** function. The first argument for this function is a driver object for the database you want to connect to. We can create a DuckDB driver object using the **duckdb::duckdb()** expression. This will be constant across all connections we make when working with DuckDB in R. The **dbdir** argument describes the target DuckDB database.

In [2]:
disk_conn <- dbConnect(
    duckdb::duckdb(),
    dbdir = "quack.duckdb"
)

In [3]:
read_only_conn <- dbConnect(
    duckdb::duckdb(),
    dbdir = "quack.duckdb",
    read_only = TRUE
)

In [4]:
mem_conn <- dbConnect(
    duckdb::duckdb(),
    dbdir = ":memory:"
)

### Reading and writing tables

In [6]:
library(tidyverse) 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


We'll use the **readr::read_csv()** function from tidyverse, which allows us to read a CSV file into a dataframe, as well as provide a way to specify how specific columns should be parsed into R data types. One notable thing about data-returning functions from tidyverse is that rather than returning base R dataframes, which are of the **data.frame** type. they return *tibbles*. These are the tidyverse's leaner and more efficient dataframes and can be used interchangeably with standard R dataframes; in addition to **tbl_df**, they also inherit from **data.frame**.

In [7]:
getwd()

[1] "/workspaces/Getting-Started-with-DuckDB/chapter_09"

In [8]:
dogs_df <- read_csv(
    file = "NYC_Dog_Licensing_Dataset.csv",
    col_types = cols(
        LicenseIssuedDate = col_date("%m/%d/%Y"),
        LicenseExpiredDate = col_date("%m/%d/%Y"),
        AnimalGender = col_factor(levels = c("M", "F")),
    ),
)

head(dogs_df, 5)

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”


AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<fct>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
PAIGE,F,2014,American Pit Bull Mix / Pit Bull Mix,10035,2014-09-12,2017-09-12,2016
YOGI,M,2010,Boxer,10465,2014-09-12,2017-10-02,2016
ALI,M,2014,Basenji,10013,2014-09-12,2019-09-12,2016
QUEEN,F,2013,Akita Crossbreed,10013,2014-09-12,2017-09-12,2016
LOLA,F,2009,Maltese,10028,2014-09-12,2017-10-09,2016


In [9]:
conn <- dbConnect(
    duckdb::duckdb(),
    dbdir = "nyc_dogs.duckdb"
)

In [10]:
dbWriteTable(
    conn,
    "dogs_table_from_df",
    dogs_df,
    overwrite = TRUE
)

In [11]:
dbListTables(conn)

[1] "dogs_table_from_df"

In [12]:
dbListFields(conn, "dogs_table_from_df")

[1] "AnimalName"         "AnimalGender"       "AnimalBirthYear"   
[4] "BreedName"          "ZipCode"            "LicenseIssuedDate" 
[7] "LicenseExpiredDate" "Extract Year"

Another frequently occurring situation is the reverse of what we just did: needing to convert a DuckDB table into a dataframe that we can work with in R. You can do this by using the **DBI::dbReadTable()** function. Let's do this now by converting our **dogs_table_from_df** table back into a dataframe. We'll inspect the
final five rows of it using the **utils::tail()** function:

In [13]:
dogs_from_duckdb_df <- dbReadTable(conn, "dogs_table_from_df")

tail(dogs_from_duckdb_df, 5)

,AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract.Year
,<chr>,<fct>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
616886,SKYE,F,2016,Great Pyrenees,11218,2023-11-01,2024-12-02,2023
616887,UNKNOWN,F,2023,Shih Tzu Crossbreed,10022,2023-11-01,2024-11-01,2023
616888,MUNYU,M,2009,"Poodle, Toy",11355,2023-11-01,2024-11-24,2023
616889,SAINT,M,2021,Unknown,11412,2023-11-01,2024-11-01,2023
616890,BABY,F,2021,Unknown,10473,2023-11-01,2024-11-01,2023


Another way we can create a table in DuckDB using the DBI package is with the **DBI::dbcreateTable()** function, which will
create a new table with a desired name and schema, but without any data. We can call this function in two ways. The first is by passing a dataframe or tibble into the third argument, which will cause DuckDB to create a new table using the column names and data types from the dataframe to create the table's schema. However, no records from the dataframe will be written:

In [14]:
dbCreateTable(conn, "empty_dogs_table", dogs_df) 

dbReadTable(conn, "empty_dogs_table") 

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract.Year
<chr>,<chr>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>


The other way we can use **DBI::dbcreateTable()** to create a new table is by specifying the column names and types directly. Rather than passing in a dataframe or tibble into the third argument, we can pass in a named character vector whose element's names and values indicate the column names and R types, respectively:

In [15]:
dbCreateTable( 
    conn,
    "good_dogs_table",
    c(name = "character", birthday = "date", age = "double")
) 

dbListTables(conn)

[1] "dogs_table_from_df" "empty_dogs_table"   "good_dogs_table"

You can confirm how R tpyes will be mapped into DuckDB data types using the **DBI::dbDataType()** funtion, which takes R objects and returns strings representing SQL data types:

In [16]:
dbDataType(conn, "Monty")

dbDataType(conn, 6)

dbDataType(conn, today())

[1] "STRING"

[1] "DOUBLE"

[1] "DATE"

Finally, we can also delete tables using the **DBI::dbRemoveTable()** function:

In [17]:
dbRemoveTable(conn, "good_dogs_table") 

dbListTables(conn)

[1] "dogs_table_from_df" "empty_dogs_table"

### Querying and executing SQL statements

In [18]:
dbGetQuery(
    conn,
    "
    SELECT ZipCode,
        count(*) AS num_registrations
    FROM dogs_table_from_df
    GROUP BY ZipCode
    ORDER BY num_registrations DESC
    LIMIT 10
    "
)

ZipCode,num_registrations
<dbl>,<dbl>
10025,13819
10023,11189
11201,10907
11215,10849
10024,10581
10011,10249
10128,10174
10009,9678
10314,8886


If you are working interactively with R, such as when performing data analysis, you will almost always want to use **DBI::dbGetQuery()** to evaluate SQL queries synchronously. However, there are times when you may want to retrieve query results asynchronously, such as when developing an application or library. For these situations, consult the DBI documentation for the **DBI::dbsendQuery()** and **DBI::dbFetch()** functions: https://dbi.r-dbi.org/reference/dbSendQuery.html.

When you need to run a SQL statement that does not return a result set, the function from the DBI package you want is **DBI::dbExecute()**. This takes the same arguments as **DBI::dbGetquery()** — a DBI connection and string containing a SQL statement — with the difference being that it does not return a dataframe. Let's have a look at a situation where we would want to use **DBI::dbExecute()**.

In [19]:
dbExecute(
    conn,
    " 
    CREATE OR REPLACE VIEW nyc_dogs_csv_view AS
    SELECT *
    FROM read_csv(
            'NYC_Dog_Licensing_Dataset.csv',
            ignore_errors=True
        )
    "
)

[1] 0

In [20]:
dbGetQuery(
    conn,
    " 
    SELECT *
    FROM nyc_dogs_csv_view
    USING SAMPLE 3
    "
)

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<chr>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
CASPER,M,2013,"Poodle, Miniature",10458,2015-03-14,2016-03-14,2016
YOGGY,M,2010,Yorkshire Terrier,11374,2015-02-22,2017-03-20,2016
MOJITO,M,2013,Poodle,10468,2017-04-11,2018-03-02,2017


To retrieve the contents of the view as a dataframe to work with this dataset in R,
we can either query the view by using a SQL query with **DBI::dbGetQuery()** or use the **DBI::dbReadTable()** function. In terms of choosing between these two strategies, **DBI::dbReadTable()** offers a simple way of reading the complete
contents of a database table or view into a dataframe, whereas **DBI::dbGetQuery()** allows you to provide a SQL query whose results will be loaded into a dataframe.

In [21]:
dogs_df1 = dbGetQuery(conn, "SELECT * FROM nyc_dogs_csv_view")

dogs_df2 = dbReadTable(conn, "nyc_dogs_csv_view")

### Using Prepared statements  

The DBI library provides support for prepared statements, which allows you to safely parameterize SQL queries and statements without exposing yourself to SQL injection vulnerabilities. Both **DBI::dbGetQuery()** and **DBI::dbExecute()** can take a second
argument. This is a list of parameters that will be matched with the corresponding **?** characters in the SQL query.

In [22]:
dbGetQuery(
    conn,
    "
    SELECT *
    FROM nyc_dogs_csv_view
    WHERE BreedName = ? AND AnimalGender = ?
    LIMIT ?
    ",
    list("Beagle", "M", 3)
)

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<chr>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
SAMMY,M,2007,Beagle,11231,2014-09-14,2017-10-15,2016
BAXTER,M,2013,Beagle,11374,2014-09-15,2019-11-21,2016
JUPLAY,M,2002,Beagle,11106,2014-09-16,2017-10-23,2016


You can also create prepared statements with placeholder slots that can be reused multiple times as needed. To do this, you'll want to use **DBI::dbSendQuery()** for queries that return result sets and **DBI::dbSendStatement()** for other statements, such as deletions, insertions, and creations. In each case, you will create a result object, after which you can use **DBI::dbBind()** to invoke a specific set of values multiple times. Once you've used this result object, you need to use **DBI::dbClearResult()** to clean up the result object.

In [23]:
result <- dbSendStatement(conn, "INSERT INTO dogs_table_from_df VALUES (?, ?, ?, ?, ?, ?, ?, ?)")

dbBind(result, list("John", "M", 2013, "Jack Russell Terrier", 10261, "2014-07-12", "2017-08-09", 2016))

dbBind(result, list("Lady Fluffina", "F", 2014, "Bichon Frisé", 10302, "2014-08-22", "2017-09-25", 2016))

dbClearResult(result)

In [24]:
tail(dbReadTable(conn, "dogs_table_from_df"), 2)

,AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract.Year
,<chr>,<fct>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
616891,John,M,2013,Jack Russell Terrier,10261,2014-07-12,2017-08-09,2016
616892,Lady Fluffina,F,2014,Bichon Frisé,10302,2014-08-22,2017-09-25,2016


### Disconnecting from DuckDB  

When you've finished working with your DuckDB database, it's important to close the connection. This is especially tme when you're writing to a persistent on-disk database, to ensure that all writes are flushed from any temporary data structures to disk. While connections are closed implicitly when they go out of scope, to avoid ambiguity and risk of data loss, it is safer to close connections explicitly using the **DBI::dbDisconnect()** function:

In [25]:
dbDisconnect(conn, shutdown = TRUE)

If you find yourself defining a function that needs to establish a connection to a DuckDB database for the duration of the function, a good practice is to set up an **exit handler**, which is an R feature that allows you to register code that will be run, regardless of whether the function ends normally or as the result of an error.
This is particularly useful when you're setting up any temporary global state, such as a database connection, that should be reset when it is no longer needed. You can do this using R's **on.exit()** function, which takes an expression as an argument that will be run when the function exits. Let's say we wanted to define a function called that takes a DuckDB database path and a SQL query string as arguments and evaluates the SQL query against the target database, returning the results as a dataframe. Inside this function, after opening a connection to the target database, we can add an exit handler that will close the connection and shut down the database on completion of the function. This is how this would look:

In [26]:
run_query <- function(db_path, sql_query) {
    conn <- dbConnect(duckdb::duckdb(), dbdir = db_path)
    on.exit(dbDisconnect(conn, shutdown = TRUE), add = TRUE)
    dbGetQuery(conn, sql_query)
}

In [27]:
run_query(
    "nyc_dogs.duckdb",
    "SELECT * FROM dogs_table_from_df USING SAMPLE 3"
)

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<fct>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
MADELINE,F,2007,Jack Russell Terrier,10075,2015-01-30,2016-02-12,2016
COCO,F,2013,Unknown,11412,2015-04-12,2016-02-11,2016
MAYA,F,2013,Golden Retriever,10065,2015-04-28,2016-06-03,2016


By passing our target database and target query into the **run_query()** function, a connection to the database is created and then automatically closed when the function returns. If we need to add further behavior to this function, we can safely extend the code without having to remember to close the connection at the end.

## Registering R objects as virtual tables 

An alternative approach to querying data from a dataframe, without needing to load it into a new table first, is to register a dataframe or tibble as a virtual table. This is similar to how a SQL view works, in that it involves creating an entry in the database catalog that points to a target dataframe, enabling DuckDB queries to be performed against the dataframe directly. We can also do the same thing with Apache Arrow tables, registering them as views within our DuckDB database's catalog. In this section, we'll provide some brief examples for registering both dataframes and Arrow tables as views that we can query.

In [28]:
conn <- dbConnect(
    duckdb::duckdb(),
    dbdir = "nyc_dogs.duckdb"
)

### Registering a dataframe as a virtual table

To register a dataframe or tibble as a virtual table, we can use the **duckdb::duckdb_register()** function.

In [29]:
duckdb::duckdb_register(conn, "dogs_df_view", dogs_df)

dbListTables(conn)

[1] "dogs_df_view"       "dogs_table_from_df" "empty_dogs_table"  
[4] "nyc_dogs_csv_view"

In [30]:
dbGetQuery(
    conn, 
    "SELECT * FROM dogs_df_view USING SAMPLE 3"
)

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<fct>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
OLIVE,F,2002,Pug,10029,2015-03-08,2016-03-08,2016
JAZZABELLE,F,2011,Beagle,11235,2014-12-26,2016-01-09,2016
DJ,M,2014,American Pit Bull Terrier/Pit Bull,10465,2015-04-28,2016-04-28,2016


If you do need to manually clear the reference to allow the dataframe to be garbage collected and free up memory, you can explicitly unregister the virtual table using **duckdb::duckdb_unregister():**

In [31]:
duckdb::duckdb_unregister(conn, "dogs_df_view")

dbListTables(conn)

[1] "dogs_table_from_df" "empty_dogs_table"   "nyc_dogs_csv_view"

### Registering an Arrow table as a virtual table

DuckDB also allows us to register Arrow tables as virtual tables. Arrow tables are in-memory column-oriented data structures that are similar to R dataframes, but since they are backed by the language-agnostic Apache Arrow format, they have the advantage
of not needing to be serialized into memory across different language clients. If you want to learn more about leveraging Apache Arrow for fast and efficient data analytics, the book *In-Memory Analytics with Apache Arrow*, also published by Packt, is an excellent resource.

In [32]:
library(arrow) 

dogs_arrow = arrow::arrow_table(dogs_df)


Attaching package: ‘arrow’


The following object is masked from ‘package:lubridate’:

    duration


The following object is masked from ‘package:utils’:

    timestamp




To register an Arrow table as a virtual table, we need to use the **duckdb::duckdb_register_arrow()** function.

In [33]:
duckdb::duckdb_register_arrow(conn, "dogs_arrow_view", dogs_arrow)

In [34]:
dbGetQuery(
    conn, 
    "SELECT * FROM dogs_arrow_view USING SAMPLE 3"
)

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<chr>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
KING,M,2013,Unknown,11206,2015-03-06,2016-03-18,2016
HONIE,F,2012,Pomeranian,11207,2015-02-22,2016-02-22,2016
HUCKLEBERRY,M,2005,Greyhound,11215,2015-02-05,2016-02-05,2016


Note that for Arrow-backed virtual tables, to discover those that have already been registered, you need to use the **duckdb::duckdb_list_arrow()** function:

In [35]:
duckdb::duckdb_list_arrow(conn)

[1] "dogs_arrow_view"

To unregister Arrow-backed virtual tables, you need to invoke the **duckdb::duckdb_unregister_arrow()** function:

In [ ]:
duckdb::duckdb_unregister_arrow(conn, "dogs_arrow_view")

## *WHAT ABOUT REPLACEMENT SCANS?*

*In Chapter 8, we saw how the DuckDB Python client allows us to leverage replacement scans to query in-memory Python data structures, including dataframes and Arrow tables, as if they were tables in a DuckDB database, simply by referencing the name of
variables in scope, and without having to explicitly register them. You might be wondering why we haven't mentioned this convenient feature in the context of the R client. At the time of writing this has not been implemented, however, work has commenced in that direction. Check the DuckDB documentation for updates!*

## Querying DuckDB with `dplyr`

The dplyr package is highly regarded among data practitioners who use R for performing data analysis and modeling. It provides users with a set of key verbs for manipulating data, such as **select, filter, arrange, summarize, and mutate**. By enabling users to combine these verbs through a composable grammar of data manipulation, the dplyr API provides an elegant and intuitive interface for constructing analytical queries programmatically. dplyr can be used to query a range of data backends, including R dataframes, Apache Arrow tables, Apache Spark datasets, and a variety of popular SQL databases. The dataframe backend is the most frequently used, allowing users to query R dataframes and tibbles using the dplyr interface. The dbplyr package provides an alternative backend that enables dplyr to be used as a query interface for a range of SQL-based databases. It works behind the scenes by translating dplyr operations into the SQL dialect of the database you have connected to. DuckDB has good support for dbplyr, which means that we can use dplyr to compose queries that are executed against DuckDB databases, without the need to write SQL.



## *DUCKPLYR*

*While this book was in the final stages of completion, a new package called duckplyr was released. duckplyr is designed to be a drop-in replacement for dplyr when querying DuckDB databases. Rather than translating dplyr expressions into DuckDB SQL, duckplyr uses a DuckDB-native interface. This offers several benefits compared to using the standard dplyr package, including cleaner integration with DuckDB, improved query performance, and better diagnostic feedback around query errors. If you find the dplyr interface for working with DuckDB compelling, you may want to try out duckplyr. We'll also discuss duckplyr in Chapter 12, as we explore the wider
DuckDB ecosystem.*

### Using `dplyr` to query dataframes

In [36]:
dogs_df |>
    group_by(LicenseIssuedYear = year(LicenseIssuedDate)) |>
    summarise(count = n()) |>
    arrange(desc(count))

LicenseIssuedYear,count
<dbl>,<int>
2020,105090
2021,87246
2016,77015
2019,75087
2017,74282
2018,72513
2023,48106
2022,41282
2015,34712


Back to our dplyr code, let's look at how it turns out that we can simplify the code for performing this analysis. Counting the occurrences of values in a table is such a common type of group- by and summarized set of operations that the **dplyr::count()**
function exists to cater to this need. Let's use it to make our preceding query more concise:

In [37]:
dogs_df |>
    count(
        LicenseIssuedYear = year(LicenseIssuedDate),
        name = "count",
        sort = TRUE
    )

LicenseIssuedYear,count
<dbl>,<int>
2020,105090
2021,87246
2016,77015
2019,75087
2017,74282
2018,72513
2023,48106
2022,41282
2015,34712


## *WHY ARE WE USING THE |> PIPE INSTEAD OF THE %>% PIPE?*

*If you've used dplyr before, you may be wondering why we're using the |> pipe operator, which is the base R pipe operator, to compose our dplyr operations, rather than the %>% pipe operator, which comes from the **magrittr** package in the tidyverse. These two pipe operators largely have the same functionality, but the base R pipe operator is a more recent addition, only being introduced in R 4.1.0. The %>% operator does support some advanced functionality beyond the base R pipe operator; however, this isn't needed for the examples in this chapter. Hence, we've chosen to use the base R |> operator.*

### Using `dplyr` to query DuckDB tables via `dbplyr`

dbplyr is an alternative backend for dplyr that translates dplyr code into a range of target SQL dialects. DuckDB has good support for dbplyr, which means that we can compose DuckDB queries using the convenient dplyr interface without the need to write SQL directly while still retaining all the performance benefits that DuckDB has over dataframes.

In [38]:
tbl(conn, "nyc_dogs_csv_view")

# Source:   table<nyc_dogs_csv_view> [?? x 8]
# Database: DuckDB 1.4.4-dev175 [unknown@Linux 6.8.0-1030-azure:R 4.3.3//workspaces/Getting-Started-with-DuckDB/chapter_09/nyc_dogs.duckdb]
   AnimalName AnimalGender AnimalBirthYear BreedName   ZipCode LicenseIssuedDate
   <chr>      <chr>                  <dbl> <chr>         <dbl> <date>           
 1 PAIGE      F                       2014 American P…   10035 2014-09-12       
 2 YOGI       M                       2010 Boxer         10465 2014-09-12       
 3 ALI        M                       2014 Basenji       10013 2014-09-12       
 4 QUEEN      F                       2013 Akita Cros…   10013 2014-09-12       
 5 LOLA       F                       2009 Maltese       10028 2014-09-12       
 6 IAN        M                       2006 Unknown       10013 2014-09-12       
 7 BUDDY      M                       2008 Unknown       10025 2014-09-12       
 8 CHEWBACCA  F                       2012 Labrador R…   10013 2014-09-12       
 9 H

The central idea behind the dbplyr backend is to use this lazy table representation as our data source in our dplyr pipeline and to construct our desired query by applying a sequence of dplyr verbs. The result of this will still be a lazy table object, but it will now include our desired query logic. To convert this table object into its corresponding DuckDB SQL and run it against the target database, we need to call the **dplyr::collect()** function against the table object we have built up. We can do this by putting it at the end of our dplyr pipeline. Let's see this in action by applying a single-head operation to our view:

In [39]:
tbl(conn, "nyc_dogs_csv_view") |>
    head(3) |> 
    collect()

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,LicenseIssuedDate,LicenseExpiredDate,Extract Year
<chr>,<chr>,<dbl>,<chr>,<dbl>,<date>,<date>,<dbl>
PAIGE,F,2014,American Pit Bull Mix / Pit Bull Mix,10035,2014-09-12,2017-09-12,2016
YOGI,M,2010,Boxer,10465,2014-09-12,2017-10-02,2016
ALI,M,2014,Basenji,10013,2014-09-12,2019-09-12,2016


We successfully retrieved a tibble that contains the results of the simple DuckDB query that we created using the dplyr interface, augmented with the dbplyr backend. To inspect the DuckDB SQL code that's generated by our query, we can replace
**dplyr::collect()** at the end of the chain with **dplyr::show_query()**:

In [40]:
tbl(conn, "nyc_dogs_csv_view") |>
    head(3) |>
    show_query()

<SQL>
SELECT nyc_dogs_csv_view.*
FROM nyc_dogs_csv_view
LIMIT 3


### Data wrangling with `dplyr`

In [41]:
unique_dogs <- tbl(conn, "nyc_dogs_csv_view") |>
    filter(!AnimalName %in% c("UNKNOWN", "NAME NOT PROVIDED", "NAME", "NONE")) |>
    filter(!BreedName %in% c("Unknown", "Not Provided")) |>
    distinct(AnimalName, AnimalGender, AnimalBirthYear, BreedName, ZipCode)

In [42]:
unique_dogs |>
    head(5) |>
    collect()

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode
<chr>,<chr>,<dbl>,<chr>,<dbl>
HILDIE,F,2016,Dachshund Smooth Coat Miniature,10464
BENTLEY,M,2014,Yorkshire Terrier,10459
CHLOE,F,2005,Morkie,11427
DEJA,F,2013,American Pit Bull Terrier/Pit Bull,10471
CARAMEL,M,2012,Shih Tzu,10468


In [43]:
table_query <- unique_dogs |>
    filter(AnimalGender == "M") |>
    count(AnimalName, name = "num_dogs", sort = TRUE) |>
    head(10)

In [44]:
table_query |>
    show_query()

<SQL>
SELECT AnimalName, COUNT(*) AS num_dogs
FROM (
  SELECT DISTINCT AnimalName, AnimalGender, AnimalBirthYear, BreedName, ZipCode
  FROM nyc_dogs_csv_view
  WHERE
    (NOT(AnimalName IN ('UNKNOWN', 'NAME NOT PROVIDED', 'NAME', 'NONE'))) AND
    (NOT(BreedName IN ('Unknown', 'Not Provided'))) AND
    (AnimalGender = 'M')
) q01
GROUP BY AnimalName
ORDER BY num_dogs DESC
LIMIT 10


In [46]:
table_query |>
    collect()

AnimalName,num_dogs
<chr>,<dbl>
MAX,2341
CHARLIE,1814
ROCKY,1600
MILO,1512
TEDDY,1452
BUDDY,1228
TOBY,1053
LEO,1047
LUCKY,1016


In [47]:
pop_dog_names <- unique_dogs |>
    filter(AnimalGender == "F", AnimalBirthYear > 2010) |>
    count(AnimalBirthYear, AnimalName, name = "NumDogs") |>
    slice_max(by = AnimalBirthYear, order_by = NumDogs) |>
    arrange(AnimalBirthYear)

In [48]:
pop_dog_names |>
    collect()

AnimalBirthYear,AnimalName,NumDogs
<dbl>,<chr>,<dbl>
2011,BELLA,146
2012,BELLA,138
2013,BELLA,169
2014,BELLA,191
2015,BELLA,198
2016,BELLA,236
2017,LUNA,236
2018,LUNA,241
2019,LUNA,251


### Calling DuckDB functions with dplyr

In [49]:
issued_by_year = tbl(conn, "nyc_dogs_csv_view") |>
    count(
        LicenseIssuedYear = year(LicenseIssuedDate),
        name = "Count",
        sort = TRUE
    )

issued_by_year |>
    collect()

LicenseIssuedYear,Count
<dbl>,<dbl>
2020,105090
2021,87246
2016,77015
2019,75087
2017,74282
2018,72513
2023,48106
2022,41282
2015,34712


In [50]:
issued_by_year |> 
    show_query()

<SQL>
SELECT LicenseIssuedYear, COUNT(*) AS Count
FROM (
  SELECT
    nyc_dogs_csv_view.*,
    EXTRACT(year FROM LicenseIssuedDate) AS LicenseIssuedYear
  FROM nyc_dogs_csv_view
) q01
GROUP BY LicenseIssuedYear
ORDER BY Count DESC


One of DuckDB's attractive features is its API's inclusion of a rich collection of functions that support a range of different analytical use cases. To make effective use of DuckDB with dplyr and dbplyr, we're going to want to be able to leverage these functions in our dplyr operation chains. Fortunately, dbplyr has us covered as it passes unknown function identifiers into the resultant generated SQL. This means that in many contexts, we can simply use DuckDB functions within our dplyr code as if they were R functions.

Let's look at an example of this in action. When working with variable-length strings, sometimes, we need to convert them into a fixed-size representation, typically an integer. This process is known as **hashing**. A commonly used hash function for converting strings into fixed-size integers is the MD5 algorithm. Let's imagine that we needed to generate a new column in our unique-dogs dataset that contains the MD5 hash of the concatenated **AnimalName** and **Zipcode** values for each record, which gives us a unique value for each pair of dog name and zip code values. DuckDB provides two functions that we can leverage for this task: **md5** and **concat**.

In [51]:
unique_dogs |>
    mutate(Hash = md5(concat(AnimalName, ZipCode))) |>
    head(3) |>
    collect()

AnimalName,AnimalGender,AnimalBirthYear,BreedName,ZipCode,Hash
<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>
FLAPJACK,M,2016,German Shepherd Crossbreed,10024,063948ff4a4491fa85863dec91fa7fd5
TIGGER,M,2003,Boston Terrier,10005,f5ede042f7d6b4de00629995a4e8b62d
MISS,F,2013,Poodle,11101,af04b1be37c689986f34958b9931ecc5


With the help of the **dplyr::mutate()** function and DuckDB's **hash** and **concat** functions, capturing this transformation in dplyr can be performed rather concisely and elegantly.

This time, we'll come up with a query that allows us to look for all dog names that are similar to a target name. As part of its extensive collection of functions, DuckDB supports a range of text-similarity functions. You can find these here: https://duckdb.org/docs/sql/functions/char.html#text-similarity-functions. We'll use the **Jaro-Winkler** distance, which provides a measure of the distance between two strings in terms of how many edits it would take to transform one into the other. This distance takes real number values from 0 to 1, with 0 indicating an exact match and 1 indicating no similarity. DuckDB makes this available via the **jaro_winkler_similarity** function, which takes two strings arguments, returning their Jaro-Winkler distance. Another DuckDB function that will come in handy is **round**, which rounds numerical values to a desired number of decimal places.

In [52]:
unique_dogs |>
    count(AnimalName, name = "Count") |>
    filter(Count >= 100) |>
    mutate(EditDistance=round(jaro_winkler_similarity(AnimalName, "BELLA"), 3)) |>
    arrange(desc(EditDistance)) |>
    head(10) |>
    collect()

AnimalName,Count,EditDistance
<chr>,<dbl>,<dbl>
BELLA,2741,1.000
ELLA,324,0.933
BELLE,259,0.920
STELLA,820,0.822
DELILAH,135,0.790
BILLY,135,0.760
ISABELLA,114,0.742
ELLIE,413,0.733
ZELDA,136,0.733


You might find it interesting to run this query against other target names and
inspect the resulting top-ranked words and their similarity scores. For more information about the Jaro-Winkler distance, its Wikipedia article offers a good overview: https://en.wikipedia.org/wiki/Jaro%E2%80%93Winkler distance.

These examples have served to further illustrate the synergistic combination of the dplyr interface and the DuckDB SQL API. They also hopefully illustrate the value of being familiar with the types of functions that can be found in the DuckDB API, so that you're ready to supercharge your analytics workflows with the right functions at the right time. You can browse the complete DuckDB function reference here: https://duckdb.org/docs/sql/functions/overview.

## Summary


With that, you know about the different ways in which you can leverage DuckDB in your R-based analytical workflows, and you're also equipped with some guiding principles for how to make effective use of them.